# TriX Quickstart

**Deterministic neural networks that never hallucinate.**

This notebook demonstrates TriX's core capability: converting Verilog hardware designs into executable neural networks using frozen polynomial shapes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anjaustin/zor/blob/main/notebooks/quickstart.ipynb)

## Setup

First, let's clone TriX and build the native ops.

In [ ]:
# Clone TriX
!git clone https://github.com/anjaustin/zor.git trix
%cd trix

# Build native ops (C with SIMD)
!cd src/trix/native/ops && make

# Add to path
import sys
sys.path.insert(0, 'src')

## The Core Idea

Boolean logic has **exact polynomial representations**:

| Gate | Polynomial |
|------|------------|
| AND | `a * b` |
| OR | `a + b - ab` |
| XOR | `a + b - 2ab` |
| NOT | `1 - a` |

These are mathematically exact on binary inputs {0, 1}. Let's verify:

In [ ]:
import numpy as np

# XOR polynomial: a + b - 2ab
def poly_xor(a, b):
    return a + b - 2*a*b

# Test all inputs
print("XOR Truth Table:")
print("a | b | a XOR b | polynomial")
print("-" * 30)
for a in [0, 1]:
    for b in [0, 1]:
        expected = a ^ b
        actual = poly_xor(a, b)
        match = "✓" if expected == actual else "✗"
        print(f"{a} | {b} |    {expected}    |     {int(actual)}     {match}")

## Using Native Ops

TriX implements these polynomials in C with SIMD acceleration (NEON on ARM, AVX2 on x86).

In [ ]:
from trix.native.ops import TrixOps

ops = TrixOps()
print(f"TriX Native Ops v{ops.version}")
print(f"SIMD Backend: {ops.simd}")

# Vectorized operations
a = np.array([0.0, 1.0, 0.0, 1.0])
b = np.array([0.0, 0.0, 1.0, 1.0])

print(f"\na = {a}")
print(f"b = {b}")
print(f"XOR(a, b) = {ops.xor(a, b)}")
print(f"AND(a, b) = {ops.multiply(a, b)}")
print(f"OR(a, b) = {ops.or_op(a, b)}")

## Importing Verilog Designs

The **Ingest** module converts Yosys-synthesized Verilog into TriX systems.

Let's load a pre-synthesized full adder:

In [ ]:
from trix.forge.ingest import ingest_yosys_json, execute, system_summary

# Load the synthesized full adder
system = ingest_yosys_json('examples/ingest/full_adder.json')

# See what we got
print(system_summary(system))

In [ ]:
# Execute the circuit
print("Full Adder Execution:")
print("a | b | cin | sum | cout")
print("-" * 25)

for a in [0, 1]:
    for b in [0, 1]:
        for cin in [0, 1]:
            result = execute(system, {'a': a, 'b': b, 'cin': cin})
            print(f"{a} | {b} |  {cin}  |  {int(result['sum'])}  |   {int(result['cout'])}")

## Exhaustive Validation

We can validate the TriX system against a reference implementation:

In [ ]:
from trix.forge.ingest import validate_exhaustive

def reference_full_adder(inputs):
    """Golden reference implementation"""
    total = inputs['a'] + inputs['b'] + inputs['cin']
    return {
        'sum': total % 2,
        'cout': total // 2
    }

passed, failures = validate_exhaustive(system, reference_full_adder)
print(f"Validation: {'PASS ✓' if passed else 'FAIL ✗'}")
print(f"Test cases: 8/8")

## 8-bit Adder (131K Test Cases)

Let's try something bigger - an 8-bit adder with 52 gates:

In [ ]:
# Load 8-bit adder
system8 = ingest_yosys_json('examples/ingest/adder8.json')
print(system_summary(system8))

In [ ]:
# Test a few cases
test_cases = [
    (0, 0, 0),
    (1, 1, 0),
    (127, 1, 0),
    (255, 1, 0),
    (255, 255, 1),
]

print("8-bit Adder Test Cases:")
print("a     | b     | cin | sum   | cout | expected")
print("-" * 50)

for a, b, cin in test_cases:
    # Convert to individual bits
    inputs = {'cin': cin}
    for i in range(8):
        inputs[f'a[{i}]'] = (a >> i) & 1
        inputs[f'b[{i}]'] = (b >> i) & 1

    result = execute(system8, inputs)

    # Reconstruct output
    sum_val = sum(int(result[f'sum[{i}]']) << i for i in range(8))
    cout = int(result['cout'])

    expected = a + b + cin
    expected_sum = expected & 0xFF
    expected_cout = (expected >> 8) & 1

    match = "✓" if (sum_val == expected_sum and cout == expected_cout) else "✗"
    print(f"{a:5d} | {b:5d} |  {cin}  | {sum_val:5d} |   {cout}  | {expected_sum:5d},{expected_cout} {match}")

## What Just Happened?

1. **Verilog** was synthesized to standard cells by Yosys
2. **TriX Ingest** converted cells to frozen polynomial shapes
3. **Native ops** executed the polynomials with SIMD acceleration
4. **Validation** proved algebraic equivalence to the original design

This is **compilation**, not learning. The "neural network" is algebraically derived from the circuit structure.

**Result:** Deterministic execution that's guaranteed correct.

## Next Steps

- **[INGEST.md](https://github.com/anjaustin/zor/blob/main/docs/INGEST.md)** - Full documentation
- **[ARCHITECTURE.md](https://github.com/anjaustin/zor/blob/main/docs/ARCHITECTURE.md)** - Deep technical reference
- **[QUICKSTART.md](https://github.com/anjaustin/zor/blob/main/QUICKSTART.md)** - More paths to explore